In [5]:
# ============================================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================================
# Core data manipulation libraries
import pandas as pd
import numpy as np
import os
from pathlib import Path
from collections import Counter, defaultdict

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Biopython for FASTA parsing

# ============================================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================================
# Core data manipulation libraries
import pandas as pd
import numpy as np
import os
from pathlib import Path
from collections import Counter, defaultdict
from Bio import SeqIO 
# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Biopython for FASTA parsing
#from Bio import SeqIO

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# Define base directory and output directory
BASE_DIR = Path('d:/cafa-5-protein-function-prediction')
OUTPUT_DIR = BASE_DIR / 'EDA_Results'
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ All libraries imported successfully")
print(f"✓ Output directory created at: {OUTPUT_DIR}")

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# Define base directory and output directory
BASE_DIR = Path('d:/cafa-5-protein-function-prediction')
OUTPUT_DIR = BASE_DIR / 'EDA_Results'
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ All libraries imported successfully")
print(f"✓ Output directory created at: {OUTPUT_DIR}")

✓ All libraries imported successfully
✓ Output directory created at: d:\cafa-5-protein-function-prediction\EDA_Results
✓ All libraries imported successfully
✓ Output directory created at: d:\cafa-5-protein-function-prediction\EDA_Results


# CAFA-5 Protein Function Prediction: Comprehensive Exploratory Data Analysis
## Complete dataset analysis including sequences, GO labels, and preprocessing recommendations

This notebook performs a full exploratory data analysis on the CAFA-5 protein function prediction dataset. It includes:
- FASTA sequence parsing and statistics
- GO term label analysis
- Data quality issue detection
- Comprehensive visualizations
- Preprocessing recommendations for deep learning models

## Section 1: Read and Parse FASTA Files

Parse training and test FASTA files using Biopython's SeqIO module. This extracts ProteinID and Sequence information into structured DataFrames.

In [4]:
from Bio import SeqIO
print("Biopython works!")

Biopython works!


In [6]:
# ============================================================================
# 2. READ AND PARSE FASTA FILES
# ============================================================================
# Install biopython if not already installed


def parse_fasta_file(fasta_path):
    """
    Parse FASTA file and return a DataFrame with EntryID and Sequence.
    
    Parameters:
    -----------
    fasta_path : str
        Path to FASTA file
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: EntryID, Sequence, SeqLength
    """
    proteins = []
    
    # Parse FASTA file using Biopython
    for record in SeqIO.parse(fasta_path, 'fasta'):
        protein_id = record.id
        sequence = str(record.seq)
        seq_length = len(sequence)
        proteins.append({
            'EntryID': protein_id,
            'Sequence': sequence,
            'SeqLength': seq_length
        })
    
    df = pd.DataFrame(proteins)
    return df

# Parse training sequences
print("Parsing training sequences...")
train_fasta_path = BASE_DIR / 'Train' / 'train_sequences.fasta'
train_sequences = parse_fasta_file(str(train_fasta_path))
print(f"✓ Train sequences loaded: {len(train_sequences)} proteins")
print(train_sequences.head())
print(f"  Shape: {train_sequences.shape}")

# Parse test sequences
print("\nParsing test sequences...")
test_fasta_path = BASE_DIR / 'Test (Targets)' / 'testsuperset.fasta'
test_sequences = parse_fasta_file(str(test_fasta_path))
print(f"✓ Test sequences loaded: {len(test_sequences)} proteins")
print(test_sequences.head())
print(f"  Shape: {test_sequences.shape}")

Parsing training sequences...
✓ Train sequences loaded: 142246 proteins
      EntryID                                           Sequence  SeqLength
0      P20536  MNSVTVSHAPYTITYHDDWEPVMSQLVEFYNEVASWLLRDETSPIP...        218
1      O73864  MTEYRNFLLLFITSLSVIYPCTGISWLGLTINGSSVGWNQTHHCKL...        354
2      O95231  MRLSSSPPRGPQQLSSFGSVDWLSQSSCSGPTHTPRPADFSLGSLP...        258
3  A0A0B4J1F4  MGGEAGADGPRGRVKSLGLVFEDESKGCYSSGETVAGHVLLEAAEP...        415
4      P54366  MVETNSPPAGYTLKRSPSDLGEQQQPPRQISRSPGNTAAYHLTTAM...        415
  Shape: (142246, 3)

Parsing test sequences...
✓ Test sequences loaded: 141865 proteins
  EntryID                                           Sequence  SeqLength
0  Q9CQV8  MTMDKSELVQKAKLAEQAERYDDMAAAMKAVTEQGHELSNEERNLL...        246
1  P62259  MDDREDLVYQAKLAEQAERYDEMVESMKKVAGMDVELTVEERNLLS...        255
2  P68510  MGDREQLLQRARLAEQAERYDDMASAMKAVTELNEPLSNEDRNLLS...        246
3  P61982  MVDREQLVQKARLAEQAERYDDMAAAMKNVTELNEPLSNEERNLLS...        247
4  O70456  MERASLIQKAKL

## Section 2: Load and Explore Labels Data

Read the training labels CSV file containing ProteinID, GO_ID, and Aspect information.

In [7]:
# ============================================================================
# 3. LOAD AND EXPLORE LABELS DATA
# ============================================================================

# Load training labels
print("Loading training labels...")
labels_path = BASE_DIR / 'Train' / 'train_terms.tsv'
train_labels = pd.read_csv(labels_path, sep='\t')
print(f"✓ Labels loaded: {len(train_labels)} label assignments")
print(f"  Columns: {train_labels.columns.tolist()}")
print("\nFirst 10 rows:")
print(train_labels.head(10))
print(f"\nDataFrame info:")
print(train_labels.info())

# Check unique aspects
print(f"\nUnique Aspects (GO categories):")
print(train_labels.groupby('aspect').size())

# Verify column names
print(f"\nColumn names: {train_labels.columns.tolist()}")

Loading training labels...
✓ Labels loaded: 5363863 label assignments
  Columns: ['EntryID', 'term', 'aspect']

First 10 rows:
      EntryID        term aspect
0  A0A009IHW8  GO:0008152    BPO
1  A0A009IHW8  GO:0034655    BPO
2  A0A009IHW8  GO:0072523    BPO
3  A0A009IHW8  GO:0044270    BPO
4  A0A009IHW8  GO:0006753    BPO
5  A0A009IHW8  GO:1901292    BPO
6  A0A009IHW8  GO:0044237    BPO
7  A0A009IHW8  GO:1901360    BPO
8  A0A009IHW8  GO:0008150    BPO
9  A0A009IHW8  GO:1901564    BPO

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5363863 entries, 0 to 5363862
Data columns (total 3 columns):
 #   Column   Dtype 
---  ------   ----- 
 0   EntryID  object
 1   term     object
 2   aspect   object
dtypes: object(3)
memory usage: 122.8+ MB
None

Unique Aspects (GO categories):
aspect
BPO    3497732
CCO    1196017
MFO     670114
dtype: int64

Column names: ['EntryID', 'term', 'aspect']


## Section 3: Merge Sequences with Labels

Combine training sequences with their corresponding GO term labels for comprehensive analysis.

In [8]:
# ============================================================================
# 4. MERGE SEQUENCES WITH LABELS
# ============================================================================

print("Merging sequences with labels...")

# Create a summary dataframe for each protein (group labels by protein)
# More efficient approach: use groupby agg instead of apply with lambda
labels_per_protein = train_labels.groupby('EntryID', as_index=False).agg(
    GO_Terms=('term', list),
    Aspects=('aspect', list),
    Num_Labels=('term', 'count')
)

# Rename the count column properly
labels_per_protein = labels_per_protein.rename(columns={'Num_Labels': 'Num_Labels'})

print(f"✓ Created labels_per_protein: {labels_per_protein.shape}")
print(f"  Columns: {labels_per_protein.columns.tolist()}")
print(f"  Sample:\n{labels_per_protein.head()}")

# Merge with sequences
print("\nMerging with sequences...")
train_merged = train_sequences.merge(labels_per_protein, on='EntryID', how='left')

print(f"✓ Merged train data shape: {train_merged.shape}")
print(f"  Total proteins with sequences: {len(train_sequences)}")
print(f"  Proteins with labels: {labels_per_protein.shape[0]}")
print(f"  Proteins without labels: {train_merged['Num_Labels'].isna().sum()}")

print("\nFirst 5 rows of merged data:")
print(train_merged[['EntryID', 'SeqLength', 'Num_Labels']].head())

Merging sequences with labels...
✓ Created labels_per_protein: (142246, 4)
  Columns: ['EntryID', 'GO_Terms', 'Aspects', 'Num_Labels']
  Sample:
      EntryID                                           GO_Terms  \
0  A0A009IHW8  [GO:0008152, GO:0034655, GO:0072523, GO:004427...   
1  A0A021WW32  [GO:0048869, GO:0048856, GO:0022008, GO:006500...   
2  A0A021WZA4  [GO:0071944, GO:0005575, GO:0110165, GO:001602...   
3  A0A023FBW4  [GO:0019956, GO:0005488, GO:0003674, GO:001995...   
4  A0A023FBW7  [GO:0019957, GO:0019956, GO:0005488, GO:000367...   

                                             Aspects  Num_Labels  
0  [BPO, BPO, BPO, BPO, BPO, BPO, BPO, BPO, BPO, ...          49  
1  [BPO, BPO, BPO, BPO, BPO, BPO, BPO, BPO, BPO, ...          77  
2                          [CCO, CCO, CCO, CCO, CCO]           5  
3                     [MFO, MFO, MFO, MFO, MFO, MFO]           6  
4                     [MFO, MFO, MFO, MFO, MFO, MFO]           6  

Merging with sequences...
✓ Merged train da

## Section 4: Compute Basic Statistics

Calculate comprehensive statistics on sequences, GO terms, and label distributions.

In [9]:
# ============================================================================
# 5. COMPUTE BASIC STATISTICS
# ============================================================================

print("=" * 80)
print("DATASET OVERVIEW STATISTICS")
print("=" * 80)

# Basic stats
print(f"\n1. PROTEIN COUNTS:")
print(f"   Train proteins (with sequences): {len(train_sequences)}")
print(f"   Test proteins (with sequences): {len(test_sequences)}")
print(f"   Train proteins with labels: {labels_per_protein.shape[0]}")
print(f"   Train proteins without labels: {len(train_sequences) - labels_per_protein.shape[0]}")

# Sequence length statistics
print(f"\n2. SEQUENCE LENGTH STATISTICS (Train Set):")
seq_stats = train_sequences['SeqLength'].describe()
print(f"   Min length: {seq_stats['min']:.0f}")
print(f"   Max length: {seq_stats['max']:.0f}")
print(f"   Mean length: {seq_stats['mean']:.1f}")
print(f"   Median length: {train_sequences['SeqLength'].median():.1f}")
print(f"   Std Dev: {seq_stats['std']:.1f}")

test_seq_stats = test_sequences['SeqLength'].describe()
print(f"\n3. SEQUENCE LENGTH STATISTICS (Test Set):")
print(f"   Min length: {test_seq_stats['min']:.0f}")
print(f"   Max length: {test_seq_stats['max']:.0f}")
print(f"   Mean length: {test_seq_stats['mean']:.1f}")
print(f"   Median length: {test_sequences['SeqLength'].median():.1f}")
print(f"   Std Dev: {test_seq_stats['std']:.1f}")

# GO term statistics
print(f"\n4. GO TERM STATISTICS:")
print(f"   Total label assignments: {len(train_labels)}")
print(f"   Unique GO terms: {train_labels['term'].nunique()}")
print(f"   Unique Aspects: {train_labels['aspect'].nunique()}")

aspect_counts = train_labels['aspect'].value_counts().sort_index()
print(f"\n   GO Term Counts by Aspect:")
for aspect, count in aspect_counts.items():
    print(f"     {aspect}: {count}")

# Labels per protein statistics
labels_per_protein['Num_Labels_clean'] = labels_per_protein['Num_Labels']
print(f"\n5. LABELS PER PROTEIN STATISTICS:")
label_dist = labels_per_protein['Num_Labels_clean'].describe()
print(f"   Min labels: {label_dist['min']:.0f}")
print(f"   Max labels: {label_dist['max']:.0f}")
print(f"   Mean labels: {label_dist['mean']:.2f}")
print(f"   Median labels: {labels_per_protein['Num_Labels_clean'].median():.0f}")
print(f"   Std Dev: {label_dist['std']:.2f}")

# Top unique GO terms
print(f"\n6. TOP 20 MOST FREQUENT GO TERMS:")
top_go_terms = train_labels['term'].value_counts().head(20)
for idx, (go_id, count) in enumerate(top_go_terms.items(), 1):
    print(f"   {idx:2d}. {go_id}: {count} occurrences")

DATASET OVERVIEW STATISTICS

1. PROTEIN COUNTS:
   Train proteins (with sequences): 142246
   Test proteins (with sequences): 141865
   Train proteins with labels: 142246
   Train proteins without labels: 0

2. SEQUENCE LENGTH STATISTICS (Train Set):
   Min length: 3
   Max length: 35375
   Mean length: 553.6
   Median length: 411.0
   Std Dev: 641.7

3. SEQUENCE LENGTH STATISTICS (Test Set):
   Min length: 2
   Max length: 35213
   Mean length: 477.1
   Median length: 375.0
   Std Dev: 462.5

4. GO TERM STATISTICS:
   Total label assignments: 5363863
   Unique GO terms: 31466
   Unique Aspects: 3

   GO Term Counts by Aspect:
     BPO: 3497732
     CCO: 1196017
     MFO: 670114

5. LABELS PER PROTEIN STATISTICS:
   Min labels: 2
   Max labels: 815
   Mean labels: 37.71
   Median labels: 24
   Std Dev: 42.52

6. TOP 20 MOST FREQUENT GO TERMS:
    1. GO:0005575: 92912 occurrences
    2. GO:0008150: 92210 occurrences
    3. GO:0110165: 91286 occurrences
    4. GO:0003674: 78637 occurrenc

## Section 5: Generate Sequence Length Visualizations

Create histogram and boxplot visualizations for analyzing sequence length distributions.

In [12]:
# ============================================================================
# 6. SEQUENCE LENGTH VISUALIZATIONS
# ============================================================================

# 1. Histogram of sequence lengths - Train Set
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_sequences['SeqLength'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Sequence Length', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Distribution of Protein Sequence Lengths (Train Set)', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(train_sequences['SeqLength'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {train_sequences["SeqLength"].mean():.0f}')
axes[0].axvline(train_sequences['SeqLength'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {train_sequences["SeqLength"].median():.0f}')
axes[0].legend()

# 2. Histogram of sequence lengths - Test Set
axes[1].hist(test_sequences['SeqLength'], bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Sequence Length', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Distribution of Protein Sequence Lengths (Test Set)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(test_sequences['SeqLength'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {test_sequences["SeqLength"].mean():.0f}')
axes[1].axvline(test_sequences['SeqLength'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {test_sequences["SeqLength"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_Sequence_Length_Histogram.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 01_Sequence_Length_Histogram.png")
plt.close()

# 3. Boxplot of sequence lengths
fig, ax = plt.subplots(figsize=(10, 6))
box_data = [train_sequences['SeqLength'], test_sequences['SeqLength']]
bp = ax.boxplot(box_data, labels=['Train', 'Test'], patch_artist=True, widths=0.6)

# Color the boxes
colors = ['steelblue', 'coral']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Sequence Length', fontsize=11)
ax.set_title('Boxplot of Protein Sequence Lengths (Train vs Test)', fontsize=12, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_Sequence_Length_Boxplot.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 02_Sequence_Length_Boxplot.png")
plt.close()

✓ Saved: 01_Sequence_Length_Histogram.png
✓ Saved: 02_Sequence_Length_Boxplot.png


## Section 6: Analyze GO Term Distribution

Create visualizations for GO term frequencies and aspect distributions.

In [13]:
# ============================================================================
# 7. GO TERM DISTRIBUTION ANALYSIS
# ============================================================================

# Get top 20 GO terms
top_20_go = train_labels['term'].value_counts().head(20)

# 1. Bar chart for top 20 GO terms
fig, ax = plt.subplots(figsize=(12, 6))
top_20_go.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black', alpha=0.8)
ax.set_xlabel('Frequency', fontsize=11)
ax.set_ylabel('GO Term ID', fontsize=11)
ax.set_title('Top 20 Most Frequent GO Terms', fontsize=12, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_Top20_GO_Terms.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 03_Top20_GO_Terms.png")
plt.close()

# 2. Aspect distribution pie chart
aspect_dist = train_labels['aspect'].value_counts()
fig, ax = plt.subplots(figsize=(10, 7))
colors_pie = ['#FF9999', '#66B2FF', '#99FF99']
wedges, texts, autotexts = ax.pie(aspect_dist.values, labels=aspect_dist.index, autopct='%1.1f%%',
                                    colors=colors_pie, startangle=90, textprops={'fontsize': 11})
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax.set_title('Distribution of GO Term Aspects (BP, MF, CC)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_Aspect_Distribution.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 04_Aspect_Distribution.png")
plt.close()

# 3. Stacked bar chart for top GO terms by Aspect
top_20_go_ids = train_labels['term'].value_counts().head(20).index
go_aspect_dist = train_labels[train_labels['term'].isin(top_20_go_ids)].groupby(['term', 'aspect']).size().unstack(fill_value=0)

# Reorder to match top 20 frequency
go_aspect_dist = go_aspect_dist.loc[top_20_go_ids]

fig, ax = plt.subplots(figsize=(12, 8))
go_aspect_dist.plot(kind='barh', stacked=True, ax=ax, color=['#FF9999', '#66B2FF', '#99FF99'], edgecolor='black', alpha=0.8)
ax.set_xlabel('Count', fontsize=11)
ax.set_ylabel('GO Term ID', fontsize=11)
ax.set_title('Top 20 GO Terms Distribution by Aspect', fontsize=12, fontweight='bold')
ax.legend(title='Aspect', loc='best')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_Top20_GO_Terms_by_Aspect.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 05_Top20_GO_Terms_by_Aspect.png")
plt.close()

✓ Saved: 03_Top20_GO_Terms.png
✓ Saved: 04_Aspect_Distribution.png
✓ Saved: 05_Top20_GO_Terms_by_Aspect.png


## Section 7: Detect Data Quality Issues

Identify potential data issues including missing sequences, duplicates, imbalanced labels, and outliers.

In [11]:
# ============================================================================
# 8. DETECT DATA QUALITY ISSUES
# ============================================================================

print("=" * 80)
print("DATA QUALITY ISSUES DETECTION")
print("=" * 80)

issues = []

# 1. Check for duplicate EntryIDs in sequences
train_seq_duplicates = train_sequences['EntryID'].duplicated().sum()
test_seq_duplicates = test_sequences['EntryID'].duplicated().sum()
print(f"\n1. DUPLICATE PROTEIN IDs:")
print(f"   Train duplicates: {train_seq_duplicates}")
print(f"   Test duplicates: {test_seq_duplicates}")
if train_seq_duplicates > 0:
    issues.append(f"Found {train_seq_duplicates} duplicate EntryIDs in train sequences")

# 2. Check for proteins with no labels
proteins_no_labels = len(train_sequences) - len(labels_per_protein)
print(f"\n2. PROTEINS WITHOUT LABELS:")
print(f"   Count: {proteins_no_labels} out of {len(train_sequences)}")
print(f"   Percentage: {proteins_no_labels/len(train_sequences)*100:.2f}%")
if proteins_no_labels > 0:
    issues.append(f"{proteins_no_labels} proteins have no GO term labels ({proteins_no_labels/len(train_sequences)*100:.1f}%)")

# 3. Check for extremely short or long sequences
short_threshold = 50
long_threshold = 5000
very_short = (train_sequences['SeqLength'] < short_threshold).sum()
very_long = (train_sequences['SeqLength'] > long_threshold).sum()
print(f"\n3. SEQUENCE LENGTH OUTLIERS:")
print(f"   Very short (<{short_threshold} aa): {very_short} proteins")
print(f"   Very long (>{long_threshold} aa): {very_long} proteins")
if very_short > 0:
    issues.append(f"{very_short} proteins are very short (<{short_threshold} aa)")
if very_long > 0:
    issues.append(f"{very_long} proteins are very long (>{long_threshold} aa)")

# 4. Check label distribution imbalance
print(f"\n4. LABEL IMBALANCE ANALYSIS:")
label_counts_per_protein = labels_per_protein['Num_Labels_clean'].value_counts().sort_index()
print(f"   Distribution of labels per protein:")
for num_labels, count in label_counts_per_protein.head(10).items():
    pct = count / len(labels_per_protein) * 100
    print(f"     {num_labels} labels: {count} proteins ({pct:.1f}%)")

# GO term frequency imbalance
print(f"\n5. GO TERM FREQUENCY IMBALANCE:")
go_freq = train_labels['term'].value_counts()
print(f"   Most frequent GO term: {go_freq.index[0]} ({go_freq.values[0]} occurrences)")
print(f"   Least frequent GO term: {go_freq.index[-1]} ({go_freq.values[-1]} occurrence)")
print(f"   Imbalance ratio: {go_freq.values[0] / go_freq.values[-1]:.1f}x")
if go_freq.values[0] / go_freq.values[-1] > 100:
    issues.append(f"Severe GO term imbalance detected ({go_freq.values[0] / go_freq.values[-1]:.0f}x ratio)")

# 6. Check for empty sequences
empty_seqs = (train_sequences['SeqLength'] == 0).sum()
print(f"\n6. EMPTY SEQUENCES:")
print(f"   Count: {empty_seqs}")
if empty_seqs > 0:
    issues.append(f"Found {empty_seqs} empty sequences")

# 7. Proteins only in test set
test_only = set(test_sequences['EntryID']) - set(train_sequences['EntryID'])
print(f"\n7. UNIQUE TEST PROTEINS:")
print(f"   Proteins only in test set: {len(test_only)} ({len(test_only)/len(test_sequences)*100:.1f}%)")

print(f"\n" + "=" * 80)
print("SUMMARY OF ISSUES DETECTED:")
print("=" * 80)
if issues:
    for i, issue in enumerate(issues, 1):
        print(f"{i}. ⚠️  {issue}")
else:
    print("✓ No critical issues detected")

DATA QUALITY ISSUES DETECTION

1. DUPLICATE PROTEIN IDs:
   Train duplicates: 0
   Test duplicates: 1

2. PROTEINS WITHOUT LABELS:
   Count: 0 out of 142246
   Percentage: 0.00%

3. SEQUENCE LENGTH OUTLIERS:
   Very short (<50 aa): 1426 proteins
   Very long (>5000 aa): 253 proteins

4. LABEL IMBALANCE ANALYSIS:
   Distribution of labels per protein:
     2 labels: 131 proteins (0.1%)
     3 labels: 9370 proteins (6.6%)
     4 labels: 3471 proteins (2.4%)
     5 labels: 4926 proteins (3.5%)
     6 labels: 2782 proteins (2.0%)
     7 labels: 3115 proteins (2.2%)
     8 labels: 3926 proteins (2.8%)
     9 labels: 5218 proteins (3.7%)
     10 labels: 4235 proteins (3.0%)
     11 labels: 2650 proteins (1.9%)

5. GO TERM FREQUENCY IMBALANCE:
   Most frequent GO term: GO:0005575 (92912 occurrences)
   Least frequent GO term: GO:0051008 (1 occurrence)
   Imbalance ratio: 92912.0x

6. EMPTY SEQUENCES:
   Count: 0

7. UNIQUE TEST PROTEINS:
   Proteins only in test set: 68211 (48.1%)

SUMMARY OF

## Section 8: Create Comprehensive Visualizations

Generate additional visualizations for label distributions and relationships.

In [14]:
# ============================================================================
# 9. COMPREHENSIVE VISUALIZATIONS
# ============================================================================

# 1. Distribution of number of labels per protein
fig, ax = plt.subplots(figsize=(12, 6))
label_dist_counts = labels_per_protein['Num_Labels_clean'].value_counts().sort_index()
ax.bar(label_dist_counts.index, label_dist_counts.values, color='steelblue', edgecolor='black', alpha=0.7, width=0.8)
ax.set_xlabel('Number of GO Labels per Protein', fontsize=11)
ax.set_ylabel('Number of Proteins', fontsize=11)
ax.set_title('Distribution of Number of Labels per Protein', fontsize=12, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

# Log scale if distribution is very skewed
if label_dist_counts.max() / label_dist_counts.min() > 100:
    ax.set_yscale('log')
    ax.set_ylabel('Number of Proteins (log scale)', fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '06_Labels_Per_Protein_Distribution.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 06_Labels_Per_Protein_Distribution.png")
plt.close()

# 2. Cumulative distribution of labels per protein
fig, ax = plt.subplots(figsize=(12, 6))
cumsum = label_dist_counts.cumsum()
cumsum_pct = (cumsum / cumsum.max() * 100)
ax.plot(label_dist_counts.index, cumsum_pct, marker='o', linewidth=2, markersize=6, color='steelblue')
ax.fill_between(label_dist_counts.index, cumsum_pct, alpha=0.3, color='steelblue')
ax.set_xlabel('Number of GO Labels per Protein', fontsize=11)
ax.set_ylabel('Cumulative Percentage of Proteins (%)', fontsize=11)
ax.set_title('Cumulative Distribution of Labels per Protein', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 105])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '07_Cumulative_Labels_Distribution.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 07_Cumulative_Labels_Distribution.png")
plt.close()

# 3. Unique GO terms per Aspect
aspect_go_counts = train_labels.groupby('aspect')['term'].nunique().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(aspect_go_counts.index, aspect_go_counts.values, color=['#FF9999', '#66B2FF', '#99FF99'], edgecolor='black', alpha=0.8, width=0.6)
ax.set_ylabel('Number of Unique GO Terms', fontsize=11)
ax.set_title('Number of Unique GO Terms by Aspect', fontsize=12, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '08_Unique_GO_Terms_by_Aspect.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 08_Unique_GO_Terms_by_Aspect.png")
plt.close()

# 4. Go frequency distribution (log scale)
go_freq = train_labels['term'].value_counts()
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(go_freq.values, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('Frequency of GO Term', fontsize=11)
ax.set_ylabel('Number of GO Terms', fontsize=11)
ax.set_title('Distribution of GO Term Frequencies (How often each GO term appears)', fontsize=12, fontweight='bold')
ax.set_yscale('log')
ax.set_xscale('log')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '09_GO_Frequency_Distribution.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 09_GO_Frequency_Distribution.png")
plt.close()

✓ Saved: 06_Labels_Per_Protein_Distribution.png
✓ Saved: 07_Cumulative_Labels_Distribution.png
✓ Saved: 08_Unique_GO_Terms_by_Aspect.png
✓ Saved: 09_GO_Frequency_Distribution.png


## Section 9: Generate Summary Report

Compile comprehensive statistics into tables and export as CSV and Markdown files.

In [10]:
# ============================================================================
# 10. GENERATE SUMMARY REPORT
# ============================================================================

print("=" * 80)
print("GENERATING COMPREHENSIVE SUMMARY REPORT")
print("=" * 80)

# 1. Create summary statistics table
summary_stats = {
    'Metric': [
        'Total Train Proteins',
        'Total Test Proteins',
        'Train Proteins with Labels',
        'Train Proteins without Labels',
        'Total Label Assignments',
        'Unique GO Terms',
        'Mean Sequence Length (Train)',
        'Median Sequence Length (Train)',
        'Min Sequence Length (Train)',
        'Max Sequence Length (Train)',
        'Mean Labels per Protein',
        'Median Labels per Protein',
        'Min Labels per Protein',
        'Max Labels per Protein',
        'Proteins with <50 aa',
        'Proteins with >5000 aa',
        'Top GO Term Frequency',
        'Least Frequent GO Term Count',
        'Unique BP Terms',
        'Unique MF Terms',
        'Unique CC Terms',
        'Total Unique Aspects'
    ],
    'Value': [
        len(train_sequences),
        len(test_sequences),
        len(labels_per_protein),
        len(train_sequences) - len(labels_per_protein),
        len(train_labels),
        train_labels['term'].nunique(),
        f"{train_sequences['SeqLength'].mean():.1f}",
        f"{train_sequences['SeqLength'].median():.0f}",
        f"{train_sequences['SeqLength'].min()}",
        f"{train_sequences['SeqLength'].max()}",
        f"{labels_per_protein['Num_Labels_clean'].mean():.2f}",
        f"{labels_per_protein['Num_Labels_clean'].median():.0f}",
        f"{labels_per_protein['Num_Labels_clean'].min()}",
        f"{labels_per_protein['Num_Labels_clean'].max()}",
        (train_sequences['SeqLength'] < 50).sum(),
        (train_sequences['SeqLength'] > 5000).sum(),
        train_labels['term'].value_counts().values[0],
        train_labels['term'].value_counts().values[-1],
        (train_labels['aspect'] == 'BPO').sum() if 'BPO' in train_labels['aspect'].values else 0,
        (train_labels['aspect'] == 'MFO').sum() if 'MFO' in train_labels['aspect'].values else 0,
        (train_labels['aspect'] == 'CCO').sum() if 'CCO' in train_labels['aspect'].values else 0,
        train_labels['aspect'].nunique()
    ]
}

summary_df = pd.DataFrame(summary_stats)
print("\nSummary Statistics Table:")
print(summary_df.to_string(index=False))

# Save as CSV
summary_df.to_csv(OUTPUT_DIR / 'Summary_Statistics.csv', index=False)
print(f"\n✓ Saved: Summary_Statistics.csv")

# Save as Markdown
summary_df.to_markdown(OUTPUT_DIR / 'Summary_Statistics.md', index=False)
print(f"✓ Saved: Summary_Statistics.md")

# 2. Top 20 GO terms statistics
top_20_go_stats = train_labels['term'].value_counts().head(20).reset_index()
top_20_go_stats.columns = ['GO_ID', 'Frequency']
top_20_go_stats['Percentage'] = (top_20_go_stats['Frequency'] / len(train_labels) * 100).round(2)
top_20_go_stats.insert(0, 'Rank', range(1, len(top_20_go_stats) + 1))

print("\nTop 20 GO Terms Frequency Table:")
print(top_20_go_stats.to_string(index=False))

top_20_go_stats.to_csv(OUTPUT_DIR / 'Top20_GO_Terms.csv', index=False)
print(f"\n✓ Saved: Top20_GO_Terms.csv")

# 3. Label distribution per protein
label_dist_table = labels_per_protein['Num_Labels_clean'].value_counts().sort_index().reset_index()
label_dist_table.columns = ['Num_Labels', 'Num_Proteins']
label_dist_table['Percentage'] = (label_dist_table['Num_Proteins'] / label_dist_table['Num_Proteins'].sum() * 100).round(2)

print("\nLabels per Protein Distribution Table (first 15 rows):")
print(label_dist_table.head(15).to_string(index=False))

label_dist_table.to_csv(OUTPUT_DIR / 'Labels_Per_Protein_Distribution.csv', index=False)
print(f"\n✓ Saved: Labels_Per_Protein_Distribution.csv")

GENERATING COMPREHENSIVE SUMMARY REPORT

Summary Statistics Table:
                        Metric   Value
          Total Train Proteins  142246
           Total Test Proteins  141865
    Train Proteins with Labels  142246
 Train Proteins without Labels       0
       Total Label Assignments 5363863
               Unique GO Terms   31466
  Mean Sequence Length (Train)   553.6
Median Sequence Length (Train)     411
   Min Sequence Length (Train)       3
   Max Sequence Length (Train)   35375
       Mean Labels per Protein   37.71
     Median Labels per Protein      24
        Min Labels per Protein       2
        Max Labels per Protein     815
          Proteins with <50 aa    1426
        Proteins with >5000 aa     253
         Top GO Term Frequency   92912
  Least Frequent GO Term Count       1
               Unique BP Terms 3497732
               Unique MF Terms  670114
               Unique CC Terms 1196017
          Total Unique Aspects       3

✓ Saved: Summary_Statistics.csv
✓ S

## Section 10: Comprehensive Analysis Report & Recommendations

Generate final summary report with key findings, data quality issues, and preprocessing recommendations for model training.

In [16]:
# ============================================================================
# 11. COMPREHENSIVE FINAL REPORT
# ============================================================================

report_text = """
================================================================================
CAFA-5 PROTEIN FUNCTION PREDICTION: COMPREHENSIVE EDA REPORT
================================================================================

EXECUTIVE SUMMARY
-----------------
This report presents a comprehensive exploratory data analysis of the CAFA-5 
protein function prediction dataset. The dataset contains training and test 
protein sequences in FASTA format, along with Gene Ontology (GO) term labels 
for the training set.

KEY FINDINGS
============

1. DATASET OVERVIEW
   - Training proteins: {train_proteins}
   - Test proteins: {test_proteins}
   - Labeled training proteins: {labeled_proteins}
   - Total label assignments: {total_labels}
   - Unique GO terms: {unique_go}
   
2. SEQUENCE STATISTICS
   
   Training Set:
   - Mean length: {train_mean:.1f} amino acids
   - Median length: {train_median:.0f} amino acids
   - Range: {train_min} - {train_max} amino acids
   - Std Dev: {train_std:.1f}
   - Very short sequences (<50 aa): {very_short}
   - Very long sequences (>5000 aa): {very_long}
   
   Test Set:
   - Mean length: {test_mean:.1f} amino acids
   - Median length: {test_median:.0f} amino acids
   - Range: {test_min} - {test_max} amino acids

3. GO TERM LABELING
   - Mean labels per protein: {labels_mean:.2f}
   - Median labels per protein: {labels_median:.0f}
   - Range: {labels_min} - {labels_max} labels
   
   Aspect Distribution (BPO/MFO/CCO):
   - BPO (Biological Process): {bpo_count} labels
   - MFO (Molecular Function): {mfo_count} labels
   - CCO (Cellular Component): {cco_count} labels
   
   Unique GO Terms by Aspect:
   - BPO: {bpo_unique}
   - MFO: {mfo_unique}
   - CCO: {cco_unique}

4. CLASS IMBALANCE ANALYSIS
   - Most frequent GO term: {top_go} ({top_go_count} occurrences)
   - Least frequent GO term: {bottom_go} ({bottom_go_count} occurrence)
   - Imbalance ratio: {imbalance_ratio:.1f}x
   - This indicates severe label imbalance that requires special handling

DATA QUALITY ISSUES
===================
{data_issues}

PREPROCESSING RECOMMENDATIONS
==============================

1. HANDLING LABEL IMBALANCE
   * Use weighted loss functions (e.g., weight inversely proportional to class frequency)
   * Consider focal loss for imbalanced multilabel classification
   * Implement class weights in training: weight = n_samples / (n_classes * n_samples_class)
   * Use stratified k-fold cross-validation for evaluation
   * Consider SMOTE-like techniques or oversampling for rare GO terms

2. SEQUENCE ENCODING & TOKENIZATION
   * Convert amino acid sequences to numerical representations:
     - One-hot encoding: 20-dimensional vectors (standard amino acids)
     - Integer encoding: Map each amino acid to 0-19
     - Embedding methods: Use pre-trained protein embeddings (e.g., ESM, ProtBERT)
   
   * Tokenization approaches:
     - Character-level: Each amino acid is a token
     - k-mer based: Use overlapping k-grams (k=3 or k=4)
     - Biological domains: Use sequence motif extraction

3. SEQUENCE PADDING & BATCHING
   * Recommended padding strategy:
     - Pad to fixed length: {target_length} (90th percentile of lengths)
     - Alternative: Dynamic padding in batches to save memory
     - Use padding masks for attention mechanisms in deep learning
   
   * Batch considerations:
     - Create batches with similar sequence lengths (save 20-30% memory)
     - Use variable-length RNNs/Transformers for efficiency
     - Consider memory vs. speed tradeoffs

4. FEATURE ENGINEERING
   * Additional sequence properties to consider:
     - Amino acid composition (% of each AA)
     - Physicochemical properties (hydrophobicity, charge, polarity)
     - Secondary structure predictions
     - Sequence complexity (Shannon entropy)
     - Codon bias metrics (if applicable)
   
   * Protein embeddings:
     - ProtBERT: BERT-based embeddings trained on protein sequences
     - ESM (Evolutionary Scale Modeling): Large-scale protein language model
     - UniRep: Unidirectional representation of proteins
     - These provide contextualized representations capturing biological information

5. DATA SPLITTING STRATEGY
   * Use stratification to maintain label distribution
   * Consider hierarchical splitting (maintain GO term aspect distribution)
   * Recommended split: 70% train, 15% validation, 15% test
   * Ensure no protein ID overlap between splits

6. MODEL ARCHITECTURE RECOMMENDATIONS
   * For this multilabel classification task:
     - CNN: Local feature extraction from sequences
     - RNN/LSTM: Capture sequential dependencies
     - Transformers: Attention-based learning (state-of-the-art)
     - Graph Neural Networks: Leverage GO term hierarchy
   
   * Combined approaches:
     - Pre-trained embedding + shallow classifier (fast, good baseline)
     - Fine-tuned language models (better performance, more computation)
     - Ensemble methods combining multiple architectures

7. HANDLING MISSING/UNLABELED DATA
   * {proteins_no_labels} training proteins have no labels (~{pct_no_labels:.1f}%)
   * Options:
     - Exclude from training (simplest)
     - Use semi-supervised learning (self-training, pseudo-labeling)
     - Transfer learning from proteins with similar sequences
     - Multi-task learning with auxiliary tasks

8. NORMALIZATION & REGULARIZATION
   * Normalization:
     - Standard scaling for input features
     - Batch normalization in deep networks
     - Layer normalization for Transformers
   
   * Regularization techniques:
     - L1/L2 regularization to prevent overfitting
     - Dropout (0.3-0.5) after dense layers
     - Early stopping based on validation metrics
     - Data augmentation (sequence masking, noise injection)

9. EVALUATION METRICS FOR MULTILABEL CLASSIFICATION
   * Recommended metrics:
     - Exact Match Accuracy: % of proteins with exactly matching predictions
     - Hamming Loss: Fraction of incorrectly predicted labels
     - Macro F1-Score: F1 averaged across all GO terms (unweighted)
     - Micro F1-Score: F1 computed globally across all predictions
     - MAP@k: Mean Average Precision at k predictions
     - AUROC by aspect (BPO/MFO/CCO)
   
   * Per-aspect evaluation to identify imbalances

NEXT STEPS
==========
1. Implement sequence encoding (recommend starting with one-hot + pre-trained embeddings)
2. Design train/val/test split with stratification
3. Build baseline model (e.g., simple MLP on embeddings)
4. Implement weighted loss functions to handle imbalance
5. Experiment with different network architectures (CNN, LSTM, Transformer)
6. Use ensemble methods for improved predictions
7. Perform ablation studies on different features/encodings
8. Evaluate on held-out test set and submit predictions

GENERATED FILES
===============
All analysis results have been saved to: EDA_Results/
- Visualizations (PNG): Sequence lengths, GO terms, distributions, imbalance analysis
- Statistics tables (CSV): Summary stats, top GO terms, label distributions
- Report files (Markdown): Detailed findings and recommendations

================================================================================
"""

# Format the report with actual values
report_text = report_text.format(
    train_proteins=len(train_sequences),
    test_proteins=len(test_sequences),
    labeled_proteins=len(labels_per_protein),
    total_labels=len(train_labels),
    unique_go=train_labels['term'].nunique(),
    
    train_mean=train_sequences['SeqLength'].mean(),
    train_median=train_sequences['SeqLength'].median(),
    train_min=train_sequences['SeqLength'].min(),
    train_max=train_sequences['SeqLength'].max(),
    train_std=train_sequences['SeqLength'].std(),
    very_short=(train_sequences['SeqLength'] < 50).sum(),
    very_long=(train_sequences['SeqLength'] > 5000).sum(),
    
    test_mean=test_sequences['SeqLength'].mean(),
    test_median=test_sequences['SeqLength'].median(),
    test_min=test_sequences['SeqLength'].min(),
    test_max=test_sequences['SeqLength'].max(),
    
    labels_mean=labels_per_protein['Num_Labels_clean'].mean(),
    labels_median=labels_per_protein['Num_Labels_clean'].median(),
    labels_min=labels_per_protein['Num_Labels_clean'].min(),
    labels_max=labels_per_protein['Num_Labels_clean'].max(),
    
    bpo_count=(train_labels['aspect'] == 'BPO').sum() if 'BPO' in train_labels['aspect'].values else 0,
    mfo_count=(train_labels['aspect'] == 'MFO').sum() if 'MFO' in train_labels['aspect'].values else 0,
    cco_count=(train_labels['aspect'] == 'CCO').sum() if 'CCO' in train_labels['aspect'].values else 0,
    
    bpo_unique=train_labels[train_labels['aspect'] == 'BPO']['term'].nunique() if 'BPO' in train_labels['aspect'].values else 0,
    mfo_unique=train_labels[train_labels['aspect'] == 'MFO']['term'].nunique() if 'MFO' in train_labels['aspect'].values else 0,
    cco_unique=train_labels[train_labels['aspect'] == 'CCO']['term'].nunique() if 'CCO' in train_labels['aspect'].values else 0,
    
    top_go=train_labels['term'].value_counts().index[0],
    top_go_count=train_labels['term'].value_counts().values[0],
    bottom_go=train_labels['term'].value_counts().index[-1],
    bottom_go_count=train_labels['term'].value_counts().values[-1],
    imbalance_ratio=train_labels['term'].value_counts().values[0] / train_labels['term'].value_counts().values[-1],
    
    data_issues='\n   '.join([f"* {issue}" for issue in issues]) if issues else "* No critical issues detected",
    
    proteins_no_labels=len(train_sequences) - len(labels_per_protein),
    pct_no_labels=(len(train_sequences) - len(labels_per_protein)) / len(train_sequences) * 100,
    
    target_length=int(np.percentile(train_sequences['SeqLength'], 90))
)

# Save the report with UTF-8 encoding
report_path = OUTPUT_DIR / 'EDA_Report.txt'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_text)

print(report_text)
print(f"\n[Saved: EDA_Report.txt]")
print(f"\n[All analysis files saved to: {OUTPUT_DIR}]")


CAFA-5 PROTEIN FUNCTION PREDICTION: COMPREHENSIVE EDA REPORT

EXECUTIVE SUMMARY
-----------------
This report presents a comprehensive exploratory data analysis of the CAFA-5 
protein function prediction dataset. The dataset contains training and test 
protein sequences in FASTA format, along with Gene Ontology (GO) term labels 
for the training set.

KEY FINDINGS

1. DATASET OVERVIEW
   - Training proteins: 142246
   - Test proteins: 141865
   - Labeled training proteins: 142246
   - Total label assignments: 5363863
   - Unique GO terms: 31466
   
2. SEQUENCE STATISTICS
   
   Training Set:
   - Mean length: 553.6 amino acids
   - Median length: 411 amino acids
   - Range: 3 - 35375 amino acids
   - Std Dev: 641.7
   - Very short sequences (<50 aa): 1426
   - Very long sequences (>5000 aa): 253
   
   Test Set:
   - Mean length: 477.1 amino acids
   - Median length: 375 amino acids
   - Range: 2 - 35213 amino acids

3. GO TERM LABELING
   - Mean labels per protein: 37.71
   - Median 

## Section 11: Additional Analysis & Code Examples

Optional advanced analysis and code templates for model development.

In [17]:
# ============================================================================
# 12. ADDITIONAL ANALYSIS & USAGE EXAMPLES
# ============================================================================

print("=" * 80)
print("ADDITIONAL ANALYSIS & CODE EXAMPLES")
print("=" * 80)

# Example 1: Compute amino acid composition
print("\n1. EXAMPLE: Amino Acid Composition")
print("-" * 40)

def compute_aa_composition(sequence):
    """Compute amino acid composition (frequency of each AA)."""
    aa_counts = Counter(sequence)
    total = len(sequence)
    composition = {aa: count/total for aa, count in aa_counts.items()}
    return composition

# Example usage
sample_sequence = train_sequences['Sequence'].iloc[0]
aa_comp = compute_aa_composition(sample_sequence)
print(f"Sample sequence length: {len(sample_sequence)}")
print(f"Top 5 amino acids: {sorted(aa_comp.items(), key=lambda x: x[1], reverse=True)[:5]}")

# Example 2: Sequence motif analysis
print("\n2. EXAMPLE: K-mer Frequency Analysis")
print("-" * 40)

def extract_kmers(sequence, k=3):
    """Extract k-mers from sequence."""
    kmers = [sequence[i:i+k] for i in range(len(sequence)-k+1)]
    kmer_counts = Counter(kmers)
    return kmer_counts

kmers_3 = extract_kmers(sample_sequence, k=3)
print(f"Total 3-mers in sample: {len(sample_sequence) - 2}")
print(f"Unique 3-mers: {len(kmers_3)}")
print(f"Top 5 3-mers: {kmers_3.most_common(5)}")

# Example 3: Compute sequence properties
print("\n3. EXAMPLE: Sequence Property Calculation")
print("-" * 40)

# Hydrophobic amino acids
hydrophobic = set('AILMFVP')
# Charged amino acids
charged = set('DEKR')

hydro_fraction = sum(1 for aa in sample_sequence if aa in hydrophobic) / len(sample_sequence)
charged_fraction = sum(1 for aa in sample_sequence if aa in charged) / len(sample_sequence)

print(f"Hydrophobic fraction: {hydro_fraction:.3f}")
print(f"Charged fraction: {charged_fraction:.3f}")
print(f"Sequence complexity (Shannon entropy): {-sum(p*np.log(p) for p in aa_comp.values() if p > 0):.3f}")

# Example 4: Label statistics per protein
print("\n4. EXAMPLE: Multi-label Statistics")
print("-" * 40)

sample_protein_id = labels_per_protein['EntryID'].iloc[0]
protein_labels = train_labels[train_labels['EntryID'] == sample_protein_id]
print(f"Sample protein: {sample_protein_id}")
print(f"Number of GO labels: {len(protein_labels)}")
print(f"GO terms:")
for idx, row in protein_labels.iterrows():
    print(f"  - {row['term']} ({row['aspect']})")

# Example 5: Create encoded sequences for deep learning
print("\n5. EXAMPLE: One-Hot Encoding for Deep Learning")
print("-" * 40)

# Create amino acid to index mapping
AA_LIST = 'ACDEFGHIKLMNPQRSTVWY'  # 20 standard amino acids
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_LIST)}

def one_hot_encode(sequence, max_length=500):
    """Convert sequence to one-hot encoding."""
    # Initialize array: (max_length, 20)
    encoded = np.zeros((max_length, len(AA_LIST)), dtype=np.float32)
    
    # Fill in one-hot vectors up to sequence length
    for i, aa in enumerate(sequence[:max_length]):
        if aa in AA_TO_IDX:
            encoded[i, AA_TO_IDX[aa]] = 1.0
    
    return encoded

# Example: encode first protein
sample_encoded = one_hot_encode(train_sequences['Sequence'].iloc[0], max_length=500)
print(f"Original sequence length: {len(train_sequences['Sequence'].iloc[0])}")
print(f"Encoded shape: {sample_encoded.shape} (length, amino_acids)")
print(f"Non-zero entries: {np.count_nonzero(sample_encoded)}")

print("\n" + "=" * 80)
print("EDA ANALYSIS COMPLETE!")
print("=" * 80)
print(f"\nAll results saved to: {OUTPUT_DIR}")
print(f"Files generated:")
print(f"  - Visualizations: 9 PNG files")
print(f"  - Statistics: 3 CSV files + Markdown")
print(f"  - Report: EDA_Report.txt")
print(f"\nReady for model development and training!")
print("=" * 80)

ADDITIONAL ANALYSIS & CODE EXAMPLES

1. EXAMPLE: Amino Acid Composition
----------------------------------------
Sample sequence length: 218
Top 5 amino acids: [('I', 0.09174311926605505), ('L', 0.08256880733944955), ('S', 0.0779816513761468), ('K', 0.0779816513761468), ('V', 0.07339449541284404)]

2. EXAMPLE: K-mer Frequency Analysis
----------------------------------------
Total 3-mers in sample: 216
Unique 3-mers: 212
Top 5 3-mers: [('SHA', 2), ('TGV', 2), ('ESP', 2), ('GVI', 2), ('MNS', 1)]

3. EXAMPLE: Sequence Property Calculation
----------------------------------------
Hydrophobic fraction: 0.394
Charged fraction: 0.225
Sequence complexity (Shannon entropy): 2.888

4. EXAMPLE: Multi-label Statistics
----------------------------------------
Sample protein: A0A009IHW8
Number of GO labels: 49
GO terms:
  - GO:0008152 (BPO)
  - GO:0034655 (BPO)
  - GO:0072523 (BPO)
  - GO:0044270 (BPO)
  - GO:0006753 (BPO)
  - GO:1901292 (BPO)
  - GO:0044237 (BPO)
  - GO:1901360 (BPO)
  - GO:000815